# Séance 1 — Audit Qualité du Dataset MyAnimeList
**Projet : AniData Lab — Sakura Analytics**

Objectif : Explorer les fichiers bruts et identifier les problèmes de qualité.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

RAW_PATH = '../data/raw/'

## 1. Chargement des fichiers CSV

In [2]:
anime_df = pd.read_csv(RAW_PATH + 'anime.csv')
ratings_df = pd.read_csv(RAW_PATH + 'rating_complete.csv')
synopsis_df = pd.read_csv(RAW_PATH + 'anime_with_synopsis.csv')

print(f'anime.csv       : {anime_df.shape[0]:,} lignes, {anime_df.shape[1]} colonnes')
print(f'rating_complete : {ratings_df.shape[0]:,} lignes, {ratings_df.shape[1]} colonnes')
print(f'with_synopsis   : {synopsis_df.shape[0]:,} lignes, {synopsis_df.shape[1]} colonnes')

anime.csv       : 17,562 lignes, 35 colonnes
rating_complete : 57,633,278 lignes, 3 colonnes
with_synopsis   : 16,214 lignes, 5 colonnes


## 2. Aperçu des données — anime.csv

In [3]:
anime_df.head(5)

,MAL_ID,Name,Score,Genres,English name,Japanese name,Type,Episodes,Aired,Premiered,...,Score-10,Score-9,Score-8,Score-7,Score-6,Score-5,Score-4,Score-3,Score-2,Score-1
0,1,Cowboy Bebop,8.78,"Action, Adventure, Comedy, Drama, Sci-Fi, Space",Cowboy Bebop,カウボーイビバップ,TV,26,"Apr 3, 1998 to Apr 24, 1999",Spring 1998,...,229170.0,182126.0,131625.0,62330.0,20688.0,8904.0,3184.0,1357.0,741.0,1580.0
1,5,Cowboy Bebop: Tengoku no Tobira,8.39,"Action, Drama, Mystery, Sci-Fi, Space",Cowboy Bebop:The Movie,カウボーイビバップ 天国の扉,Movie,1,"Sep 1, 2001",Unknown,...,30043.0,49201.0,49505.0,22632.0,5805.0,1877.0,577.0,221.0,109.0,379.0
2,6,Trigun,8.24,"Action, Sci-Fi, Adventure, Comedy, Drama, Shounen",Trigun,トライガン,TV,26,"Apr 1, 1998 to Sep 30, 1998",Spring 1998,...,50229.0,75651.0,86142.0,49432.0,15376.0,5838.0,1965.0,664.0,316.0,533.0
3,7,Witch Hunter Robin,7.27,"Action, Mystery, Police, Supernatural, Drama, ...",Witch Hunter Robin,Witch Hunter ROBIN (ウイッチハンターロビン),TV,26,"Jul 2, 2002 to Dec 24, 2002",Summer 2002,...,2182.0,4806.0,10128.0,11618.0,5709.0,2920.0,1083.0,353.0,164.0,131.0
4,8,Bouken Ou Beet,6.98,"Adventure, Fantasy, Shounen, Supernatural",Beet the Vandel Buster,冒険王ビィト,TV,52,"Sep 30, 2004 to Sep 29, 2005",Fall 2004,...,312.0,529.0,1242.0,1713.0,1068.0,634.0,265.0,83.0,50.0,27.0


In [4]:
anime_df.dtypes

MAL_ID           int64
Name               str
Score              str
Genres             str
English name       str
Japanese name      str
Type               str
Episodes           str
Aired              str
Premiered          str
Producers          str
Licensors          str
Studios            str
Source             str
Duration           str
Rating             str
Ranked             str
Popularity       int64
Members          int64
Favorites        int64
Watching         int64
Completed        int64
On-Hold          int64
Dropped          int64
Plan to Watch    int64
Score-10           str
Score-9            str
Score-8            str
Score-7            str
Score-6            str
Score-5            str
Score-4            str
Score-3            str
Score-2            str
Score-1            str
dtype: object

In [5]:
anime_df.describe(include='all')

,MAL_ID,Name,Score,Genres,English name,Japanese name,Type,Episodes,Aired,Premiered,...,Score-10,Score-9,Score-8,Score-7,Score-6,Score-5,Score-4,Score-3,Score-2,Score-1
count,17562.000000,17562,17562,17562,17562,17562,17562,17562,17562,17562,...,17562,17562,17562,17562,17562,17562,17562,17562,17562,17562
unique,NaN,17558,533,5034,6831,16679,7,201,11947,231,...,3379,3645,4515,4933,4236,3288,2235,1506,1110,1084
top,NaN,Maou Gakuin no Futekigousha: Shijou Saikyou no...,Unknown,Hentai,Unknown,Unknown,TV,1,Unknown,Unknown,...,4.0,Unknown,Unknown,2.0,Unknown,Unknown,Unknown,Unknown,Unknown,4.0
freq,NaN,3,5141,969,10565,48,4996,8381,309,12817,...,936,3167,1371,610,511,584,977,1307,1597,955
mean,21477.192347,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,14900.093170,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,5953.500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,22820.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,35624.750000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Valeurs manquantes

In [6]:
def audit_missing(df, name):
    missing = df.isnull().sum()
    pct = (missing / len(df) * 100).round(2)
    result = pd.DataFrame({'manquants': missing, 'pourcentage': pct})
    result = result[result['manquants'] > 0].sort_values('pourcentage', ascending=False)
    print(f'\n=== {name} ===')
    print(result if not result.empty else 'Aucune valeur manquante')

audit_missing(anime_df, 'anime.csv')
audit_missing(ratings_df, 'rating_complete.csv')
audit_missing(synopsis_df, 'anime_with_synopsis.csv')


=== anime.csv ===
Aucune valeur manquante

=== rating_complete.csv ===
Aucune valeur manquante

=== anime_with_synopsis.csv ===
           manquants  pourcentage
sypnopsis          8         0.05


## 4. Doublons

In [7]:
print(f'Doublons anime.csv       : {anime_df.duplicated().sum()}')
print(f'Doublons rating_complete : {ratings_df.duplicated().sum()}')
print(f'Doublons synopsis        : {synopsis_df.duplicated().sum()}')

Doublons anime.csv       : 0


Doublons rating_complete : 0
Doublons synopsis        : 0


## 5. Types incohérents

In [8]:
# Colonnes qui devraient être numériques
print('=== Score anime ===')
print(anime_df['Score'].value_counts().head(10))
print('\nValeurs non-numériques dans Score :')
non_num = anime_df[pd.to_numeric(anime_df['Score'], errors='coerce').isna()]['Score'].unique()
print(non_num)

=== Score anime ===
Score
Unknown    5141
6.48         74
6.31         72
6.3          72
6.52         71
6.45         70
6.65         68
6.19         65
6.6          65
6.07         65
Name: count, dtype: int64

Valeurs non-numériques dans Score :
<StringArray>
['Unknown']
Length: 1, dtype: str


In [9]:
# Vérification des épisodes
print('Valeurs non-numériques dans Episodes :')
non_num_ep = anime_df[pd.to_numeric(anime_df['Episodes'], errors='coerce').isna()]['Episodes'].unique()
print(non_num_ep)

Valeurs non-numériques dans Episodes :
<StringArray>
['Unknown']
Length: 1, dtype: str


## 6. Encodages — Titres japonais

In [10]:
# Titres avec caractères non-ASCII
has_japanese = anime_df['Name'].str.contains(r'[^\x00-\x7F]', na=False)
print(f'Titres avec caractères non-ASCII : {has_japanese.sum()}')
print('\nExemples :')
print(anime_df[has_japanese]['Name'].head(10).tolist())

Titres avec caractères non-ASCII : 501

Exemples :
['Rozen Maiden: Träumend', 'Shin Shirayuki-hime Densetsu Prétear', 'Aishiteruze Baby★★', 'Onegai☆Teacher', 'Onegai☆Twins', 'Ranma ½', 'Happy☆Lesson', 'Happy☆Lesson (TV)', 'Happy☆Lesson: Advance', 'Happy☆Lesson: The Final']


## 7. Distribution des ratings

In [11]:
print('Distribution des ratings :')
print(ratings_df['rating'].value_counts().sort_index())
print(f'\nRange : {ratings_df["rating"].min()} - {ratings_df["rating"].max()}')

Distribution des ratings :
rating
1       333419
2       405556
3       696048
4      1455102
5      3436250
6      6849293
7     13325549
8     14642156
9      9773857
10     6716048
Name: count, dtype: int64



Range : 1 - 10


## 8. Résumé — Rapport d'audit

In [12]:
print('='*50)
print('RAPPORT D\'AUDIT — AniData Lab')
print('='*50)
print()
print('PROBLÈMES IDENTIFIÉS :')
print('1. Score et Episodes contiennent "Unknown" (str) → à convertir en NaN puis float/int')
print('2. Genres, Studios, Producers sont des listes en string → à parser')
print('3. Aired contient des dates en format texte → à normaliser')
print('4. Titres japonais/coréens en UTF-8 → à gérer lors de l\'indexation')
print('5. Ratings -1 dans certains datasets → valeur sentinelle à remplacer par NaN')
print()
print('ACTIONS PRÉVUES (séance 2) :')
print('- Remplacer "Unknown" par NaN')
print('- Convertir Score, Episodes en float/int')
print('- Parser les colonnes multi-valeurs (genres, studios)')
print('- Feature engineering : score_popularité, ratio_completion')
print('- Export dataset gold en CSV + JSON')

RAPPORT D'AUDIT — AniData Lab

PROBLÈMES IDENTIFIÉS :
1. Score et Episodes contiennent "Unknown" (str) → à convertir en NaN puis float/int
2. Genres, Studios, Producers sont des listes en string → à parser
3. Aired contient des dates en format texte → à normaliser
4. Titres japonais/coréens en UTF-8 → à gérer lors de l'indexation
5. Ratings -1 dans certains datasets → valeur sentinelle à remplacer par NaN

ACTIONS PRÉVUES (séance 2) :
- Remplacer "Unknown" par NaN
- Convertir Score, Episodes en float/int
- Parser les colonnes multi-valeurs (genres, studios)
- Feature engineering : score_popularité, ratio_completion
- Export dataset gold en CSV + JSON
